# Pandas Query

Il metodo query() di pandas permette di filtrare i dati di un DataFrame usando una sintassi simile a quella SQL (Structured Query Language, linguaggio usato per interagire coi database), semplice e leggibile.

In [ ]:
# Importing Libraries
import pandas as pd
import numpy as np
from datasets import load_dataset
import matplotlib.pyplot as plt  

# Carica il dataset
dataset = load_dataset("yiqing111/Engineering_Jobs_Insight_Dataset")
# Converte in DataFrame Pandas
df = dataset['train'].to_pandas()
# Rimpiazza gli spazi con l'underscore
df.columns = df.columns.str.replace(' ', '_')
# Convertire 'Date_Posted' in datetime senza specificare il formato esatto
df['Date_Posted'] = pd.to_datetime(df['Date_Posted'], errors='coerce')


In [ ]:
df.head(5)

Mostrami gli stipendi minimi maggiori di 400000

In [ ]:
df.query("Salary_Min > 400000")

# Pandas `melt()`

Il metodo melt() di pandas serve per trasformare un DataFrame "largo" in uno "lungo" — cioè passare da colonne multiple a due colonne: una con i nomi delle variabili e una con i valori.


In [ ]:
df_short = pd.DataFrame({
    'Data': ['2024-01', '2024-02'],
    'A': [100, 150],
    'B': [120, 130]
})
df_short

In [ ]:
df_short.melt(id_vars = 'Data', var_name = 'Prodotto', value_name = 'Vendite')

# Pandas Index Management

## Proprietà dell'indice

* Puoi ottenere o impostare le proprietà dell'indice, come il suo nome o il tipo di dato.  
* È utile per mantenere i metadati o garantire la compatibilità dell'indice nelle operazioni.
* `(index.name, index.dtype)`
    * `index.name` - nome dell'indice
    * `index.dtype` - tipo di dato dell'indice


In [ ]:
df.sample(3)

In [ ]:
df.index

In [ ]:
df.index.dtype

Il nostro indice è un intervallo di numeri, stiamo ispezionando il nome...

In [ ]:
df.index.name

L'indice non ha un nome, possiamo darglielo

In [ ]:
df.index.name = 'job_index'

In [ ]:
df.index.name

In [ ]:
df.sample(3)

### Esempio 2

Ricordi la tabella pivot che abbiamo creato nell'ultimo esempio? Quella in cui abbiamo ottenuto gli stipendi  minimi mediani annuali per i diversi titoli di lavoro. Eseguiamo di nuovo quel codice. Poi otterremo il nome dell'indice e il tipo di dato dell'indice usando `df.index.name` e `df.index.dtype`.


In [ ]:
median_pivot = df.pivot_table(values='Salary_Min', index='Job_Title',aggfunc='median')
median_pivot

In [ ]:
index_name = median_pivot.index.name  
index_name 

In [ ]:
index_dtype = median_pivot.index.dtype  
index_dtype

## reset_index()

* `reset_index()`: Reimposta l'indice del DataFrame al valore predefinito (intero). Questo è particolarmente utile dopo operazioni che modificano l'indice, come l'ordinamento o il filtraggio, per semplificare ulteriori manipolazioni dei dati.


### Esempio 1

Quando creiamo nuovi DataFrame filtrando, l'indice viene incasinato!


In [ ]:
df_usa = df[df['Location'] == 'US']

df_usa.head(5)

In [ ]:
df_usa.index

L'indice non è più correttamente incrementato di 1.

In [ ]:
df_usa.reset_index(inplace=True)
df_usa.head()

Tecnicamente potremmo usare `.drop()` per eliminare `job_index`.

MA, se in futuro volessimo eseguire operazioni di merge con il nostro DataFrame originale, questo fornisce l'`id` univoco per farlo.

### Esempio 2

Torniamo al nostro DataFrame principale con le offerte di lavoro. Andremo effettivamente a reimpostare gli indici nella tabella pivot così che `Job_Title` non sia più l'indice.


In [ ]:
median_pivot.reset_index(inplace=True)
median_pivot

## set_index()

* `set_index()`: Imposta una o più colonne esistenti come indice del DataFrame. Questo è utile per dati temporali (timeseries) o quando si vuole indicizzare in base a specifici attributi.


### Esempio 1

E se volessimo tornare a usare `job_index` come indice principale?


In [ ]:
df_usa.head()

In [ ]:
df_usa.set_index('job_index', inplace=True)

df_usa.head()

### Esempio 2

Ora che abbiamo reimpostato il nostro indice, possiamo impostare un nuovo indice su un'altra colonna come `Job_Title`.


In [ ]:
median_pivot.set_index('Job_Title', inplace=True)
median_pivot

## sort_index()


* `sort_index()`: Ordina il DataFrame in base all'indice (etichette delle righe), in ordine crescente o decrescente. Questo aiuta a organizzare rapidamente i dati per indice ed è spesso usato dopo `set_index()`.


### Esempio

Torniamo al nostro DataFrame pivotato e ordiniamo alfabeticamente questo nuovo indice.

In [ ]:
median_pivot.sort_index(inplace=True)
median_pivot.head(30)

# Pandas Merge DataFrames

* `merge()`: Combina DataFrame basandosi su colonne o indici comuni
* Unisce le righe dei DataFrame in base a chiavi specifiche.

## Esempi


Dobbiamo combinare due DataFrame:
- `df_jobs` contiene informazioni simili al nostro DataFrame `df`
- `df_companies` contiene informazioni sulle aziende


In [ ]:
job_data = {
    'job_id': [1, 2, 3, 4, 5],
    'job_title': ['Data Scientist', 'Software Engineer', 'Product Manager', 'Marketing Director', 'HR Manager'],
    'company_name': ['Google', 'Microsoft', 'Apple', 'Nike', 'Starbucks'],
    'job_location': ['Mountain View, CA', 'Redmond, WA', 'Cupertino, CA', 'Beaverton, OR', 'Seattle, WA']
}

company_data = {
    'company_name': ['Google', 'Microsoft', 'Apple', 'Nike', 'Starbucks'],
    'industry': ['Technology', 'Technology', 'Technology', 'Apparel', 'Food & Beverage'],
    'company_size': ['100,000+', '100,000+', '100,000+', '75,000+', '346,000+']
}


df_jobs = pd.DataFrame(job_data)
df_companies = pd.DataFrame(company_data)

In [ ]:
df_jobs

In [ ]:
df_companies

In [ ]:
# Merge the two datasets on the 'job_id' column
df_job_company = df_jobs.merge(df_companies, on='company_name')

df_job_company

Mergiamo due data frame riguardo a quando sono richiesti mensilmente i lavori

In [ ]:
import random

# Set seed for reproducibility
random.seed(42)

# Define job titles
job_titles = ['Front-End Developer', 'Back-End Developer', 'Full-Stack Developer', 'UI/UX Designer']

# Define months
months = ['January', 'February', 'March', 'April', 'May', 'June', 'July', 'August', 'September', 'October', 'November', 'December']

# Define ranges for random data
ranges = {
    'Front-End Developer': (11000, 15000),
    'Back-End Developer': (8000, 10000),
    'Full-Stack Developer': (5000, 7500),
    'UI/UX Designer': (4000, 5000)
}

# Generate random data
data = {}
for job_title in job_titles:
    data[job_title] = [random.randint(ranges[job_title][0], ranges[job_title][1]) for _ in months]

# Create DataFrame
df_US_software_pivot = pd.DataFrame(data, index=months)
df_US_software_pivot.index.name = "job_posted_month"

df_US_software_pivot

In [ ]:
import random

# Set seed for reproducibility
random.seed(12)

# Define job titles
job_titles =  ['Data Scientist', 'Software Engineer', 'Product Manager', 'Marketing Director', 'HR Manager']

# Define months
months = ['January', 'February', 'March', 'April', 'May', 'June', 'July', 'August', 'September', 'October', 'November', 'December']

# Define ranges for random data
ranges = {
    'Data Scientist': (9000, 11000),
    'Software Engineer': (8000, 10000),
    'Product Manager': (10000, 13000),
    'Marketing Director': (11000, 15000),
    'HR Manager':(14000, 18000)
}

# Generate random data
data = {}
for job_title in job_titles:
    data[job_title] = [random.randint(ranges[job_title][0], ranges[job_title][1]) for _ in months]

# Create DataFrame
df_US_data_pivot = pd.DataFrame(data, index=months)
df_US_data_pivot.index.name = "job_posted_month"

df_US_data_pivot

Combiniamoli usando merge:

In [ ]:
df_US_merged = df_US_data_pivot.merge(df_US_software_pivot, on='job_posted_month')
df_US_merged

In [ ]:
# prendiamo i primi 5
top_5 = df_US_merged.sum().sort_values(ascending=False).head(5).index

top_5 = top_5.tolist()

top_5


In [ ]:
df_US_merged[top_5].plot(kind='line')
plt.title('Monthly Job Postings for Top Tech Jobs in the US')
plt.xlabel('2023')
plt.ylabel('Job Count')
plt.legend()
plt.show()

# Pandas Concat DataFrames

* `concat()`: Combina DataFrame per righe (`axis=0`) o per colonne (`axis=1`).
* Unisce i DataFrame senza usare chiavi.

### Esempio 1

Questo è un semplice DataFrame con dati di due offerte di lavoro. La prima è di gennaio, la seconda di febbraio. Useremo `concat()` per concatenare questi due DataFrame.


In [ ]:
# Sample dataset of job postings in January
job_postings_jan = pd.DataFrame({
    'job_id': [1, 2, 3, 4, 5],
    'job_title': ['Data Scientist', 'Data Analyst', 'Machine Learning Engineer', 'Data Scientist', 'Data Engineer'],
    'company': ['Company A', 'Company B', 'Company C', 'Company D', 'Company E'],
    'job_posted_date': pd.to_datetime(['2024-01-02', '2024-01-07', '2024-01-14', '2024-01-19', '2024-01-24'])
})

job_postings_jan

In [ ]:
# Dataset di esempio con offerte di lavoro a febbraio  
job_postings_feb = pd.DataFrame({
    'job_id': [6, 7, 8, 9, 10],
    'job_title': ['Data Scientist', 'Data Analyst', 'Machine Learning Engineer', 'Data Scientist', 'Data Engineer'],
    'company': ['Company F', 'Company G', 'Company H', 'Company I', 'Company J'],
    'job_posted_date': pd.to_datetime(['2024-02-05', '2024-02-09', '2024-02-12', '2024-02-18', '2024-02-22'])
})

job_postings_feb

In [ ]:
# Concatenare i due DataFrame
# Di default axis = 0
job_postings_combined = pd.concat([job_postings_jan, job_postings_feb], ignore_index=True)

job_postings_combined

# Pandas Exporting Data

* `to_csv()`: Esporta il DataFrame in un file CSV.  
* `to_excel()`: Esporta il DataFrame in un file Excel.  
* `to_sql()`: Esporta il DataFrame in un database SQL.  
* `to_parquet()`: Esporta il DataFrame in un file Parquet.  
    * Parquet è un formato di archiviazione colonnare progettato per uno storage e un recupero efficienti dei dati.

Per prima cosa, esportiamo il nostro file in un **file CSV**.  
Spesso può essere necessario esportare e importare dati da e verso i DataFrame in formato CSV, soprattutto quando si vuole **pulire i dati** prima di utilizzarli in altri strumenti di visualizzazione.


In [ ]:
# Saving the DataFrame to a CSV file
df.to_csv('data/jobs_data.csv', index=False)

In [ ]:
help(df.to_csv)

Un altro formato molto usato per l'esportazione, soprattutto per i tuoi amici che non usano il codice, è **Excel**.


In [ ]:
# Saving the DataFrame to an Excel file
df.to_excel('jobs_data.xlsx', index=False)

In [ ]:
# Carica il dataset
dataset = load_dataset("yiqing111/Engineering_Jobs_Insight_Dataset")
# Converte in DataFrame Pandas
df = dataset['train'].to_pandas()
# Rimpiazza gli spazi con l'underscore
df.columns = df.columns.str.replace(' ', '_')
df.to_excel('data/jobs_data.xlsx', index=False)

Puoi esportare Dataframe in formato  **pickle file**.

Un file Pickle è un modo per salvare oggetti Python (come DataFrame, dizionari, liste, modelli ML, ecc.) su disco, in modo che possano essere ricaricati esattamente come erano.

In [ ]:
# Saving the DataFrame to a Pickle file
df.to_pickle('data/job_data.pkl')

# Pandas `apply()`

La funzione `apply()` permette di **applicare una funzione** personalizzata a **ciascuna colonna o riga** di un DataFrame, oppure a una **Serie**.

### Quando usarla

- Per eseguire **trasformazioni complesse**
- Per applicare **funzioni personalizzate**
- Quando il semplice `.map()` o operazioni vettoriali non bastano

---

### Sintassi base

```python
df.apply(funzione, axis=0)  # per colonne (default), lavora su tutte le righe di una colonna
df.apply(funzione, axis=1)  # per righe, lavora su tutte le colonne di una riga


### Esempio 1

Calcola gli stipendi previsti per il prossimo anno rispetto al salario minimo, usando un tasso ipotetico del 3,0% per tutti i ruoli.


In [ ]:
def inflation(salary):
    return salary * 1.03

df['salary_year_inflated'] = df['Salary_Min'].apply(inflation)

df[['Salary_Min', 'salary_year_inflated']]

In realtà possiamo semplificare questo usando una funzione lambda.

In [ ]:
df['salary_year_inflated'] = df['Salary_Min'].apply(lambda salary: salary * 1.03)

df[['Salary_Min', 'salary_year_inflated']]

Tecnicamente, questo si poteva anche fare così...

In [ ]:
df['salary_year_inflated'] = df['Salary_Min'] * 1.03

df[['Salary_Min', 'salary_year_inflated']]

### Example 2

Calcola gli stipendi previsti per il prossimo anno, ma:
- Per i ruoli senior 'Product Manager  Social app startup', considera un aumento del 400%
- Per tutti gli altri ruoli, considera un aumento del 3%


In [ ]:
def projected_salary(row):
    if 'Product Manager  Social app startup' in row['Job_Title']:
        return  4.0 * row['Salary_Min']
    else:
        return  1.03 * row['Salary_Min']

df['salary_year_inflated'] = df.apply(projected_salary, axis=1)

df[pd.notna(df['Salary_Min'])][['Job_Title', 'Salary_Min', 'salary_year_inflated']].sort_values(by='salary_year_inflated', ascending=False).head(30)

Tecnicamente potresti scriverlo con una funzione lambda, ci provi?

# Pandas Explode

* `explode()` - trasforma ogni elemento di tipo lista in una riga separata  
* Espande i dati di tipo lista all'interno di una colonna del DataFrame in righe distinte.  
* Usato spesso per analizzare dati contenuti in liste (*hint hint*) o quando si lavora con dati JSON in un DataFrame.


In [ ]:
data = {
    'job_title_short': ['Data Analyst', 'Data Scientist', 'Data Engineer'],
    'job_skills': [['excel', 'sql', 'python'], ['python', 'r'], ['aws', 'python', 'airflow']]
}

df_skills = pd.DataFrame(data)

df_skills

Se volessimo analizzare le competenze in questo caso, ci servirebbero almeno 5 righe di codice per ottenere il conteggio delle skill.

In [ ]:
df_skill_lists = df_skills.copy()

for row in df_skill_lists.itertuples():
    for skill in row.job_skills:
        df_skill_lists[skill] = df_skill_lists['job_skills'].apply(lambda x: skill in x)
        df_skill_lists[skill] = df_skill_lists[skill].astype(int)
        
df_skill_lists.loc['Total'] = df_skill_lists.sum()
    
df_skill_lists.iloc[:, 2:]

Tuttavia, esplodere i dati li rende molto più facili da gestire.

In [ ]:
df_exploded = df_skills.explode('job_skills')

df_exploded

Ora possiamo usare `value_counts` e persino creare un grafico.

In [ ]:
df_exploded.value_counts('job_skills')

In [ ]:
df_exploded.value_counts('job_skills').plot(kind='bar')